In [1]:
# -*- coding: utf-8 -*-
"""
Created on Mon Nov 21 14:31:48 2022

@author: Li Shuai
"""
#Python 3.9.12 
import numpy as np  #1.22.4
import matplotlib.pyplot as plt  #3.5.1
import seaborn as sns  #0.11.2
from shapely.geometry import MultiPoint, Point, Polygon  #1.8.2
from scipy.spatial import Voronoi #1.7.3
import shapely.geometry #1.8.2
import pandas as pd  #1.4.2
from sklearn.preprocessing import LabelEncoder  #1.0.2


def voronoi_finite_polygons_2d(vor, radius=None):

    if vor.points.shape[1] != 2:
        raise ValueError("Requires 2D input")

    new_regions = []
    new_vertices = vor.vertices.tolist()

    center = vor.points.mean(axis=0)
    if radius is None:
        radius = np.ptp(vor.points).max()

    # Construct a map containing all ridges for a given point
    all_ridges = {}
    for (p1, p2), (v1, v2) in zip(vor.ridge_points, vor.ridge_vertices):
        all_ridges.setdefault(p1, []).append((p2, v1, v2))
        all_ridges.setdefault(p2, []).append((p1, v1, v2))

    # Reconstruct infinite regions
    for p1, region in enumerate(vor.point_region):
        vertices = vor.regions[region]

        if all(v >= 0 for v in vertices):
            # finite region
            new_regions.append(vertices)
            continue

        # reconstruct a non-finite region
        ridges = all_ridges[p1]
        new_region = [v for v in vertices if v >= 0]

        for p2, v1, v2 in ridges:
            if v2 < 0:
                v1, v2 = v2, v1
            if v1 >= 0:
                # finite ridge: already in the region
                continue

            # Compute the missing endpoint of an infinite ridge

            t = vor.points[p2] - vor.points[p1] # tangent
            t /= np.linalg.norm(t)
            n = np.array([-t[1], t[0]])  # normal

            midpoint = vor.points[[p1, p2]].mean(axis=0)
            direction = np.sign(np.dot(midpoint - center, n)) * n
            far_point = vor.vertices[v2] + direction * radius

            new_region.append(len(new_vertices))
            new_vertices.append(far_point.tolist())

        # sort region counterclockwise
        vs = np.asarray([new_vertices[v] for v in new_region])
        c = vs.mean(axis=0)
        angles = np.arctan2(vs[:,1] - c[1], vs[:,0] - c[0])
        new_region = np.array(new_region)[np.argsort(angles)]

        # finish
        new_regions.append(new_region.tolist())

    return new_regions, np.asarray(new_vertices)


path_to_data = r'D:\Qiu Xinyao\CODEX\LS\20220628 celltype voronoi\testing_allReg_seurat_._analysis_20220324_AllSubtypes_withoutartifact_addct.csv'

path_to_data = r'/home/qyyuan/project/Lifei-Spatial/analysis/Part_2_discovery_cohort_CN_definition/cells_r=50_CN=10.csv'

cells2 = pd.read_csv(path_to_data)
cells2['neighborhood10']=cells2['neighborhood10'].values.astype(str)
celltype_map={'0':0, 
              '1':1,
              '2':2,
              '3':3, 
              '4':4, 
              '5':5,
              '6':6,
              '7':7, 
              '8':8,
              '9':9}

celltype_anno=['0',
              '1',
              '2',
              '3',
              '4',
              '5',
              '6',
              '7',
              '8',
              '9'     
              
              
              
              
              
              
              ]

class_=cells2['Class'].unique()
color_code=["#8358CF", # B cells-
         "#D083FF", # CD4+ T cells
         "#27A7FA", # CD8+ T cells
         "#62FFFF", # Macrophages
         "#74F882", # Fibroblasts
         "#F6D026", # Endothelial cells
         "#EE6600", # Lymphatic endothelial cells
         "#AE2519", # Biliary tract cells
         "#FA339A", # Tumor cells
         "#D083FF" # CD4+ T cell
         ]
label_code=['A','B','C','D','E','F','G','H','I','J']
voronoi_hue = 'neighborhood10'
X = 'X'
Y = 'Y'
edge_color = 'facecolor'
line_width = .1
alpha = 1
figsize = (30,30)


for m in class_:
    spot = cells2[cells2['Class']==m]
    spot['neighborhood10'] = spot['neighborhood10'].map(celltype_map).values.astype(int) 
    labels  = [label_code[i] for i in spot[voronoi_hue]]
    colors  = [color_code[i] for i in spot[voronoi_hue]]
    points=spot[[X,Y]].values
    size_max=4000 
    points[:,1] = max(points[:,1])-points[:,1]
    vor = Voronoi(points)
    label=[]
    regions, vertices = voronoi_finite_polygons_2d(vor)
    pts = MultiPoint([Point(i) for i in points])
    mask = pts.convex_hull
    new_vertices = []
    if type(alpha)!=list:
        alpha = [alpha]*len(points)
    areas = []
    plt.figure(figsize=figsize)
    for i,(region,alph) in enumerate(zip(regions,alpha)):
        polygon = vertices[region]
        shape = list(polygon.shape)
        shape[0] += 1
        p = Polygon(np.append(polygon, polygon[0]).reshape(*shape)).intersection(mask)
        areas+=[p.area]
        if p.area <size_max:
            poly = np.array(list(zip(p.boundary.coords.xy[0][:-1], p.boundary.coords.xy[1][:-1])))
            new_vertices.append(poly)
            if edge_color == 'facecolor':
                plt.fill(*zip(*poly), alpha=alph,edgecolor= 'black',linewidth = line_width , facecolor = colors[i])
                plt.text(poly.mean(0)[0],poly.mean(0)[1],s=labels[i],fontsize=5)
            else:
                plt.fill(*zip(*poly), alpha=alph,edgecolor=  edge_color,linewidth = line_width, facecolor = colors[i])
    mu=0
    for color in color_code:
        plt.scatter([],[],c=color,s=30,label=label_code[mu]+':'+celltype_anno[mu])
        mu+=1
    font = {'family': 'Arial', 'weight': 'normal', 'size': 20}
    plt.title(m, fontsize = 75)
    plt.axis('off')
    plt.savefig('..\\voronoi neighborhood10\\{}.tif'.format(str(m)))
    plt.show() 








ModuleNotFoundError: No module named 'numpy'

In [12]:
!pip3 install sklearn

  Using cached sklearn-0.0.post12.tar.gz (2.6 kB)
  Preparing metadata (setup.py) ... error
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> [15 lines of output]
      The 'sklearn' PyPI package is deprecated, use 'scikit-learn'
      rather than 'sklearn' for pip commands.
      
      Here is how to fix this error in the main use cases:
      - use 'pip install scikit-learn' rather than 'pip install sklearn'
      - replace 'sklearn' by 'scikit-learn' in your pip requirements files
        (requirements.txt, setup.py, setup.cfg, Pipfile, etc ...)
      - if the 'sklearn' package is used by one of your dependencies,
        it would be great if you take some time to track which package uses
        'sklearn' instead of 'scikit-learn' and report it to their issue tracker
      - as a last resort, set the environment variable
        SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL=True to avoid this error
      
    

In [3]:
voronoi_hue = 'celltype'

In [48]:
cells2['celltype'].unique()
 

array(['6', '1', '5', '13', '8', '4', '14', '15', '3', '16', '9', '10',
       '11', '12', '17', '2', '7'], dtype=object)

In [40]:
m

'mADT-1.tsv'

In [16]:
path_to_data = r'/home/qyyuan/project/Lifei-Spatial/analysis/Part_2_discovery_cohort_CN_definition/Allsubtypes.csv'
cells2 = pd.read_csv(path_to_data)

In [30]:
cells2



,index,X,Y,Class,celltype
0,1,910.22,116.45,mADT-1.tsv,6
1,2,922.00,126.27,mADT-1.tsv,1
2,3,961.76,139.50,mADT-1.tsv,6
3,4,959.32,143.55,mADT-1.tsv,6
4,5,1004.20,143.23,mADT-1.tsv,5
...,...,...,...,...,...
1048570,1048571,160.41,666.46,N-25.tsv,1
1048571,1048572,156.05,666.96,N-25.tsv,1
1048572,1048573,296.59,667.52,N-25.tsv,1
1048573,1048574,316.61,668.35,N-25.tsv,1


In [2]:
import pandas as pd

# 读取 CSV 文件
file_path = '/home/qyyuan/project/Lifei-Spatial/analysis/Part_2_discovery_cohort_CN_definition/Allsubtypes.csv'

# 读取文件（假设已经有正确的列名）
df = pd.read_csv(file_path, sep='\t')

print("="*50)
print("转换前：")
print("="*50)
print("列名:", df.columns.tolist())
print("\n数据类型:")
print(df.dtypes)
print("\ncelltype列的前几行:")
print(df['celltype'].head(10))
print(f"celltype列的数据类型: {df['celltype'].dtype}")

# 将 celltype 列转换为字符串类型
# 方法1：使用 astype(str)
df['celltype'] = df['celltype'].astype(str)

# 方法2：如果想去掉小数点（将 6.0 变成 "6"），可以使用：
# df['celltype'] = df['celltype'].astype(int).astype(str)

# 方法3：更灵活的处理，保留原始格式
# df['celltype'] = df['celltype'].apply(lambda x: str(int(x)) if x.is_integer() else str(x))

print("\n" + "="*50)
print("转换后：")
print("="*50)
print("数据类型:")
print(df.dtypes)
print("\ncelltype列的前几行:")
print(df['celltype'].head(10))
print(f"celltype列的数据类型: {df['celltype'].dtype}")

# 验证转换后的数据
print("\n验证转换结果:")
for i in range(min(10, len(df))):
    original_value = pd.read_csv(file_path, sep='\t')['celltype'].iloc[i]
    new_value = df['celltype'].iloc[i]
    print(f"行 {i+1}: {original_value} ({type(original_value)}) -> {new_value} ({type(new_value)})")

# 保存文件
df.to_csv(file_path, sep='\t', index=False)
print(f"\n✅ 文件已保存到: {file_path}")

转换前：
列名: ['X,Y,Class,celltype']

数据类型:
X,Y,Class,celltype    object
dtype: object

celltype列的前几行:


KeyError: 'celltype'

In [18]:
cells2['celltype']=str(cells2['celltype'].values)

In [ ]:
import pandas as pd

# 读取 CSV 文件
file_path = '/home/qyyuan/project/Lifei-Spatial/analysis/Part_2_discovery_cohort_CN_definition/Allsubtypes.csv'
df = pd.read_csv(file_path)

# 检查当前列名
print("当前列名:", df.columns.tolist())

# 将最后一列重命名为 'celltype'
df.columns = list(df.columns[:-1]) + ['celltype']

# 或者使用更直接的方法：通过位置修改最后一列名
# df.rename(columns={df.columns[-1]: 'celltype'}, inplace=True)

# 保存修改后的文件（可以选择覆盖原文件或保存为新文件）
df.to_csv(file_path, index=False)  # 覆盖原文件
# df.to_csv('/home/qyyuan/project/Lifei-Spatial/analysis/Part_2_discovery_cohort_CN_definition/Allsubtypes_with_celltype.csv', index=False)  # 保存为新文件

print("已更新列名，最后一列现在名为 'celltype'")
print("更新后的列名:", df.columns.tolist())

当前列名: ['X', 'Y', 'Class', 'celltype']


In [1]:
import pandas as pd

# 读取 CSV 文件
file_path = '/home/qyyuan/project/Lifei-Spatial/analysis/Part_2_discovery_cohort_CN_definition/Allsubtypes.csv'

# 读取时不将任何列作为索引
df = pd.read_csv(file_path, header=None, skiprows=0)

print("数据形状:", df.shape)
print("前几行数据:")
print(df.head())

# 假设数据有5列：索引 + X + Y + Class + Allsubtypes
# 设置列名
df.columns = ['index', 'X', 'Y', 'Class', 'celltype']

# 如果不需要保留索引列
df = df[['X', 'Y', 'Class', 'celltype']]

print("\n最终列名:", df.columns.tolist())
print("\n最终数据预览:")
print(df.head())

# 保存文件
df.to_csv(file_path, index=False)
print(f"\n文件已保存到: {file_path}")

/tmp/ipykernel_601653/1657317055.py:7: DtypeWarning: Columns (0,1,3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, header=None, skiprows=0)


数据形状: (774983, 4)
前几行数据:
        0       1           2         3
0       X       Y       Class  celltype
1  910.22  116.45  mADT-1.tsv         6
2   922.0  126.27  mADT-1.tsv         1
3  961.76   139.5  mADT-1.tsv         6
4  959.32  143.55  mADT-1.tsv         6


ValueError: Length mismatch: Expected axis has 4 elements, new values have 5 elements

In [2]:
import pandas as pd

# 读取 CSV 文件（制表符分隔）
file_path = '/home/qyyuan/project/Lifei-Spatial/analysis/Part_2_discovery_cohort_CN_definition/Allsubtypes.csv'

# 读取文件，指定制表符分隔，不将第一列作为索引
df = pd.read_csv(file_path, sep='\t', header=None)

print("数据形状:", df.shape)
print("前几行数据:")
print(df.head())

# 设置列名
df.columns = ['index', 'X', 'Y', 'Class', 'celltype']

print("\n设置列名后的数据:")
print(df.head())

# 验证列名
print("\n列名:", df.columns.tolist())
print("数据类型:")
print(df.dtypes)

# 保存文件，保留索引列但不保存DataFrame的索引
df.to_csv(file_path, sep='\t', index=False)
print(f"\n文件已保存到: {file_path}")

数据形状: (774983, 1)
前几行数据:
                            0
0          X,Y,Class,celltype
1  910.22,116.45,mADT-1.tsv,6
2   922.0,126.27,mADT-1.tsv,1
3   961.76,139.5,mADT-1.tsv,6
4  959.32,143.55,mADT-1.tsv,6


ValueError: Length mismatch: Expected axis has 1 elements, new values have 5 elements